# Regression - K-Nearest Neighbors

So far we have used OLS, Ridge, and Lasso for regression - all parametric linear models that learn a fixed set of coefficients. Many other models can be used for regression as well: tree-based methods like Random Forests and Gradient Boosting (Unit 4), support vector machines, and neural networks, among others. Before we move on to classification, we take a brief look at K-Nearest Neighbors, our first non-parametric model. Instead of learning parameters, KNN stores the training data and predicts by averaging nearby observations. The slides covered how the algorithm works and its strengths and limitations.

This notebook focuses on implementation, tuning, and two new sklearn tools: `validation_curve` for hyperparameter sweeps and `GridSearchCV` for multi-parameter search. We also complete the baseline hierarchy from the previous lecture.

## Setup

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

We continue with the Boston dataset.

In [ ]:
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/Boston.csv"
boston = pd.read_csv(url)

X = boston.loc[:, 'zn':]
y = boston['crim']

## KNN in a Pipeline

KNN uses distances between observations, so features on larger scales dominate the calculation. This is the same reason we scale for regularization, and the same `Pipeline` solution. The pattern is identical to Ridge:

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

pipe_knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=5)
)

scores = cross_val_score(pipe_knn, X, y, cv=cv, scoring='r2')
print(f"KNN (k=5)")
print(f"Mean CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")

`cross_val_score` returns scores computed on the held-out test fold each time - never on the training data. Throughout this notebook, "CV R²" always means test R².

To see why scaling matters, compare with and without `StandardScaler`:

In [ ]:
pipe_unscaled = make_pipeline(KNeighborsRegressor(n_neighbors=5))

scores_unscaled = cross_val_score(pipe_unscaled, X, y, cv=cv, scoring='r2')

print(f"With scaling:    Mean CV R² = {scores.mean():.4f} (+/- {scores.std():.4f})")
print(f"Without scaling: Mean CV R² = {scores_unscaled.mean():.4f} (+/- {scores_unscaled.std():.4f})")

Without scaling, features with large numeric ranges dominate the distance calculation so *the model effectively ignores features measured on smaller scales*. Distance-based methods always need scaling.

## Visualizing KNN Predictions

KNN predictions are hard to visualize with more than two or three features, but low-dimensional examples show clearly what the algorithm is doing. We use `lstat` (% lower status population) as a single predictor for the first two plots.

Note: These visualizations use one or two features for clarity. The full model uses all 12 features - the neighborhoods live in 12-dimensional space, which is why we rely on cross-validation scores rather than visual inspection to evaluate performance.

In [ ]:
x_1d = boston[['lstat']].values

# generate synthetic prediction points, evenly distributed in the training range
x_sorted = np.linspace(x_1d.min(), x_1d.max(), 300).reshape(-1, 1)

### Effect of k on the prediction curve

Each curve below shows KNN predictions across the range of `lstat` for a different value of k. At k=1 the model interpolates every training point exactly, producing a jagged curve. As k increases, the predictions smooth out because each prediction averages more neighbors. This is the bias-variance tradeoff made visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, k in zip(axes, [1, 5, 20]):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(x_1d, y)
    y_pred = knn.predict(x_sorted)

    ax.scatter(x_1d, y, alpha=0.3, s=10, color='gray', label='Training data')
    ax.plot(x_sorted, y_pred, color='C0', linewidth=2, label=f'KNN (k={k})')
    ax.set_xlabel('lstat')
    ax.set_title(f'k = {k}')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('crim')
fig.suptitle('KNN Regression: Effect of k on Predictions', y=1.02)
plt.tight_layout()
plt.show()

As $k$ increases, so does the averaging / smoothing effect.

### How a single prediction is formed

To see the mechanism behind a single prediction, we pick a query point and highlight its k nearest neighbors. The prediction is just their average (uniform weights) or distance-weighted average. With k=3, a few nearby points dominate; with k=15, the neighborhood is much wider and the prediction is more averaged out.

In [ ]:
query_x = np.array([[15.0]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, k in zip(axes, [3, 15]):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(x_1d, y)
    y_pred = knn.predict(query_x)[0]

    distances, indices = knn.kneighbors(query_x)
    neighbor_x = x_1d[indices[0]].flatten()
    neighbor_y = y.values[indices[0]]

    ax.scatter(x_1d, y, alpha=0.2, s=10, color='gray', label='Training data')
    ax.scatter(neighbor_x, neighbor_y, color='C1', s=40, zorder=3, label=f'{k} neighbors')
    for nx, ny in zip(neighbor_x, neighbor_y):
        ax.plot([nx, query_x[0, 0]], [ny, y_pred], color='C1', alpha=0.3, linewidth=0.8)
    ax.axhline(y_pred, color='C3', linestyle='--', alpha=0.6)
    ax.scatter(query_x, y_pred, color='C3', s=100, marker='*', zorder=4,
               label=f'Prediction = {y_pred:.2f}')

    ax.set_xlabel('lstat')
    ax.set_title(f'k = {k}')
    ax.set_xlim(14, 16)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('crim')
fig.suptitle('KNN: How Neighbors Determine the Prediction (zoomed in)', y=1.02)
plt.tight_layout()
plt.show()

### Prediction surface with two features

With two features we can visualize the full prediction surface as a heatmap.

In [ ]:
feat_cols = ['lstat', 'medv']
X_2d = boston[feat_cols].values

scaler = StandardScaler()
X_2d_sc = scaler.fit_transform(X_2d)

knn_2d = KNeighborsRegressor(n_neighbors=10)
knn_2d.fit(X_2d_sc, y)

# create prediction grid in original scale, then scale for prediction
grid_x = np.linspace(X_2d[:, 0].min(), X_2d[:, 0].max(), 200)
grid_y = np.linspace(X_2d[:, 1].min(), X_2d[:, 1].max(), 200)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = np.column_stack([xx.ravel(), yy.ravel()])
grid_sc = scaler.transform(grid_points)

zz = knn_2d.predict(grid_sc).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
contour = ax.contourf(xx, yy, zz, levels=3, cmap='YlOrRd', alpha=0.8)
ax.contour(xx, yy, zz, levels=3, colors='k',
  linewidths=0.3, alpha=0.5)
ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='YlOrRd', edgecolors='k',
           linewidth=0.3, s=15, alpha=0.7)

# annotate the boundary between low and moderate crime predictions
boundary = ax.contour(xx, yy, zz, levels=[10], colors='black', linewidths=2)
ax.clabel(boundary, fmt='crim = %1.0f', fontsize=10)

fig.colorbar(contour, ax=ax, label='Predicted crim')
ax.set_xlabel('lstat')
ax.set_ylabel('medv')
ax.set_title('KNN Prediction Surface (k=10, 2 features)')
plt.tight_layout()
plt.show()

The background color shows the KNN prediction at each location; training points are overlaid with their actual crime rates. The bold "crim = 10" contour marks the boundary between the low-crime region (pale, upper-left) and moderate-to-high crime (orange/red, bottom). This boundary curves and jags to follow the local density of training points. KNN carves out localized regions shaped by whichever 10 neighbors happen to be closest.

Compare this with OLS on the same two features - the prediction surface is a plane, so all boundaries are straight lines:

In [ ]:
from sklearn.linear_model import LinearRegression

ols_2d = LinearRegression()
ols_2d.fit(X_2d_sc, y)
zz_ols = ols_2d.predict(grid_sc).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
contour = ax.contourf(xx, yy, zz_ols, levels=3, cmap='YlOrRd', alpha=0.8)
ax.contour(xx, yy, zz_ols, levels=3, colors='k', linewidths=0.3, alpha=0.5)
ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='YlOrRd', edgecolors='k',
           linewidth=0.3, s=15, alpha=0.7)
boundary = ax.contour(xx, yy, zz_ols, levels=[10], colors='black', linewidths=2)
ax.clabel(boundary, fmt='crim = %1.0f', fontsize=10)
fig.colorbar(contour, ax=ax, label='Predicted crim')
ax.set_xlabel('lstat')
ax.set_ylabel('medv')
ax.set_title('OLS Prediction Surface (2 features)')
plt.tight_layout()
plt.show()

The OLS boundary is a straight diagonal line - the only shape a linear model can produce in two-dimensional feature space. It cannot capture the localized pockets of high crime that KNN finds.

## Tuning k

The default k=5 is a reasonable starting point, but not necessarily the best choice. As the slides discussed, k controls the bias-variance tradeoff: small k overfits (low averaging effect → high complexity, high variance), large k underfits (oversimplified → low complexity, high bias). We can find the best k empirically by evaluating a range of values with cross-validation.

First we'll do this manually, with a loop that fits and scores models with the specified k-values, and saves the results.

In [ ]:
k_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100]
results = []

for k in k_values:
    pipe = make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=k)
    )
    
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    results.append({'k': k, 'mean_r2': scores.mean(), 'std_r2': scores.std()})

results_df = pd.DataFrame(results)
results_df

The following plot shows the bias-variance tradeoff from the slides, measured empirically.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(results_df['k'], results_df['mean_r2'], 'o-')
ax.fill_between(
    results_df['k'],
    results_df['mean_r2'] - results_df['std_r2'],
    results_df['mean_r2'] + results_df['std_r2'],
    alpha=0.2
)

ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('Mean CV R²')
ax.set_title('KNN Regression: k vs Cross-Validated R²')
ax.grid(True, alpha=0.3)

best_idx = results_df['mean_r2'].idxmax()
best_k = results_df.loc[best_idx, 'k']
best_r2 = results_df.loc[best_idx, 'mean_r2']
ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Best k: {best_k} (Mean CV R²: {best_r2:.4f})")

Note the x-axis direction: moving right increases k, which *decreases* flexibility. Compare this with the ISL figure 2.17 from the slides, where flexibility is plotted as 1/K.

## The `weights` Parameter

By default, all k neighbors contribute equally to the prediction (the simple mean). The `weights` parameter changes this:

- `'uniform'` (default): all neighbors count equally
- `'distance'`: each neighbor is weighted by the inverse of its distance, so closer neighbors have more influence

Distance weighting makes intuitive sense - a neighbor that is very close should matter more than one at the edge of the neighborhood. It often helps when k is larger, because it prevents distant neighbors from diluting the prediction.

In [ ]:
results_uniform = results_df.copy()
results_uniform['weights'] = 'uniform'

results_distance = []
for k in k_values:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k, weights='distance'))
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    results_distance.append({
        'k': k, 'mean_r2': scores.mean(), 'std_r2': scores.std(), 'weights': 'distance'
    })

results_distance_df = pd.DataFrame(results_distance)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(results_uniform['k'], results_uniform['mean_r2'], 'o-', label='uniform')
ax.plot(results_distance_df['k'], results_distance_df['mean_r2'], 's--', label='distance')

ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('Mean CV R²')
ax.set_title('KNN: Uniform vs Distance Weighting')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

Distance weighting tends to be more forgiving of larger k values since distant neighbors are downweighted rather than ignored. The best combination of k and weights is what we want to find - which leads to the next tool.

## `validation_curve`

The manual loop above works, but sklearn provides `validation_curve` for exactly this pattern. It varies a single hyperparameter, runs cross-validation at each value, and returns both training and test scores. The training scores are useful because they reveal overfitting directly: a large gap between train and test means the model memorizes rather than generalizes.

We switch to `weights='distance'` here to see an interesting training score pattern:

In [ ]:
from sklearn.model_selection import validation_curve

pipe = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(weights='distance')
)

k_range = np.arange(1, 51)

train_scores, test_scores = validation_curve(
    pipe, X, y,
    param_name="kneighborsregressor__n_neighbors",
    param_range=k_range,
    cv=cv,
    scoring='r2'
)

The `param_name` uses a double-underscore convention to reach inside the pipeline: `kneighborsregressor__n_neighbors` means "the `n_neighbors` parameter of the `kneighborsregressor` step." The step name comes from `make_pipeline`, which lowercases the class name.

Both `train_scores` and `test_scores` have shape (n_param_values, n_cv_folds), i.e. each row holds the $R^2$ values for each fold.

In [ ]:
test_scores[-10:]

We average across folds to plot.

In [ ]:
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(k_range, train_mean, label='Training score')
ax.fill_between(k_range, train_mean - train_std, train_mean + train_std, alpha=0.1)

ax.plot(k_range, test_mean, label='Validation score')
ax.fill_between(k_range, test_mean - test_std, test_mean + test_std, alpha=0.1)

ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('R²')
ax.set_title('Validation Curve: KNN with Distance Weighting')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Why is the training $R^2$ exactly 1.0 for *every* value of $k$?

With `weights='distance'`, each training point is its own nearest neighbor at distance 0. The weight is $1/d$, so that self-neighbor gets infinite weight, and the prediction equals the true value regardless of how many other neighbors participate. With `weights='uniform'`, only k=1 would give perfect training R² - at k=5, the prediction is the average of the point and its 4 nearest neighbors, which is not exact.

The test score tells the real story. It starts low at small k (overfitting) and rises as k increases, peaking where the model best balances flexibility and stability. The train-test gap is a more informative diagnostic than test score alone - our manual loop only showed the test side.

## `GridSearchCV`

`validation_curve` varies one parameter at a time and is designed for diagnostic plotting. To *select* the best value automatically, we use `GridSearchCV`. It works for single-parameter searches too (just a grid with one key), but its real power is searching multiple parameters simultaneously.

To tune k *and* weights together, we define a grid of all combinations. `GridSearchCV` runs cross-validation on every combination and picks the best. The parameter naming uses the same double-underscore convention as `validation_curve`.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())

param_grid = {
    'kneighborsregressor__n_neighbors': [1, 2, 3, 5, 7, 10, 15, 20, 30, 50],
    'kneighborsregressor__weights': ['uniform', 'distance'],
}

grid_search = GridSearchCV(pipe, param_grid, cv=cv, scoring='r2', return_train_score=True)
grid_search.fit(X, y)

In [ ]:
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV R²:      {grid_search.best_score_:.4f}")

`GridSearchCV` evaluated 10 x 2 = 20 combinations, each with 5-fold CV, for 100 total fits. The full results are stored in `cv_results_`:

In [ ]:
results_grid = pd.DataFrame(grid_search.cv_results_)
results_grid[['param_kneighborsregressor__n_neighbors',
              'param_kneighborsregressor__weights',
              'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(10)

By default, `GridSearchCV` refits the best model on the full dataset after the search completes. The refitted model is available as `best_estimator_` and can be used directly for predictions:

In [ ]:
best_pipe = grid_search.best_estimator_
best_pipe

In [ ]:
# predict on first 5 training observations - used here to demonstrate the API
predictions = best_pipe.predict(X[:5])
pd.DataFrame({'actual': y[:5].values, 'predicted': predictions})

The values we are tuning - k and weights - are *hyperparameters*: settings chosen before training that control how the model learns, rather than values learned from the data. Ridge's alpha is a hyperparameter; Ridge's coefficients are parameters. The distinction matters because hyperparameters must be selected by an outer loop (cross-validation) while parameters are optimized by the model itself during fitting.

This is the same pipeline we would build manually, but with the best hyperparameters already set. In practice, you would split into train/test first, run `GridSearchCV` on the training set, then evaluate `best_estimator_` on the held-out test set for a final performance estimate.

Here is what `GridSearchCV` does under the hood when the estimator is a pipeline. Compare this with the Pipeline pseudocode from the previous lecture - the only addition is the outer parameter loop:

```text
For each combination of hyperparameters in the grid:
  Split X into n equal chunks
  For each fold i = 1..n (n-1 chunks train, 1 chunk test):
    1. Set the pipeline's hyperparameters for this combination
    2. Fit StandardScaler on X_train_i          ← learns mean, std from train only
    3. Transform X_train_i using that scaler
    4. Fit KNN on scaled X_train_i, y_train_i
    5. Transform X_test_i using the same scaler  ← no refit!
    6. Score KNN on scaled X_test_i, y_test_i
  Record mean score across n folds for this combination
Select the combination with the best mean score
Refit the full pipeline on all of X with those hyperparameters
```

The outer loop over the parameter grid is the only addition over plain cross-validation. Inside each combination, the pipeline handles scaling exactly as it would with `cross_val_score` - no information leaks between folds, and no information leaks between parameter evaluations.

## Completing the Baseline Hierarchy

We now have the tools to compare KNN against the linear models from the previous lecture. We use the best KNN configuration from the grid search.

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.dummy import DummyRegressor

best_params = grid_search.best_params_
best_k = best_params['kneighborsregressor__n_neighbors']
best_w = best_params['kneighborsregressor__weights']

models = {
    'Dummy (mean)': make_pipeline(DummyRegressor(strategy='mean')),
    'OLS':          make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (CV)':   make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'Lasso (CV)':   make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42)),
    f'KNN (k={best_k}, {best_w})': make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=best_k, weights=best_w)
    ),
}

print(f"{'Model':<25} {'Mean CV R²':>11} {'Std':>8}")
print("-" * 46)

for name, pipe in models.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    print(f"{name:<25} {scores.mean():>11.4f} {scores.std():>8.4f}")

### Reading the Hierarchy

The baseline hierarchy is a diagnostic tool. The *gaps* between adjacent models reveal dataset characteristics:

- *Dummy → OLS*: Does the data have any linear signal? A large gap means features contain useful information. A small gap suggests weak predictors or a non-linear relationship.
- *OLS → Ridge*: Does regularization help? A large improvement means collinear predictors with unstable OLS coefficients. Similar scores mean collinearity is not a major problem.
- *Ridge → Lasso*: Does sparsity help? If Lasso matches Ridge with fewer features, some predictors are redundant. If Lasso is worse, all features contribute and the L1 penalty is too aggressive.
- *Linear → KNN*: Is there non-linearity? If KNN outperforms the best linear model, the data has patterns that linear models cannot capture. If KNN is similar or worse, the relationship is approximately linear and the interpretability of Ridge/Lasso is preferable.

This is not a decision flowchart - it is a diagnostic panel. You read the gaps to understand your data, not to pick a winner.

## Summary

This lecture introduced KNN regression and two sklearn tools for hyperparameter tuning:

- `KNeighborsRegressor` in a Pipeline with `StandardScaler` follows the same pattern as Ridge and Lasso. Scaling is essential for distance-based methods.
- The `weights` parameter (`'uniform'` vs `'distance'`) controls how neighbors contribute to predictions. Distance weighting is often more forgiving of larger k values.
- `validation_curve` sweeps a single hyperparameter and returns train and test scores. The train-test gap reveals overfitting directly.
- `GridSearchCV` searches a grid of multiple hyperparameters simultaneously, selects the best combination via CV, and refits on the full training set. The double-underscore naming convention (`kneighborsregressor__n_neighbors`) accesses parameters inside a Pipeline.
- The baseline hierarchy (Dummy → OLS → Ridge → Lasso → KNN) is a diagnostic panel. The gaps between models reveal whether features are useful, collinear, redundant, or non-linearly related to the target.
- KNN is susceptible to the curse of dimensionality: as the number of features grows, distances become less meaningful and performance degrades. This is one reason the baseline hierarchy compares KNN against linear models rather than relying on KNN alone.

In the next lecture we will use this hierarchy as a debugging toolkit, applying it to datasets with planted problems to practice diagnosing common modeling issues.